# 🚀 TraderPath TFT Model Training

Bu notebook Google Colab Pro'da TFT (Temporal Fusion Transformer) model eğitimi için kullanılır.

**Gereksinimler:**
- Google Colab Pro (T4/A100 GPU)
- ~16GB RAM
- ~10GB disk

In [ ]:
# GPU kontrolü
!nvidia-smi

In [ ]:
# Dependencies yükle
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q lightning>=2.0.0 pytorch-forecasting>=1.1.0
!pip install -q pandas numpy httpx loguru optuna pyarrow

In [ ]:
# GitHub repo clone (veya Google Drive mount)
!git clone https://github.com/traderpath/traderpath.git /content/traderpath
%cd /content/traderpath/services/tft-predictor

In [ ]:
# Alternatif: Google Drive mount
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/traderpath/services/tft-predictor

In [ ]:
import sys
sys.path.insert(0, '/content/traderpath/services/tft-predictor/src')

import asyncio
import torch
from loguru import logger

# GPU check
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A")

## 📊 Konfigürasyon

In [ ]:
from training.config import TrainingConfig, TradeType, TRADE_TYPE_CONFIGS

# Trade type seç
TRADE_TYPE = TradeType.SWING  # SCALP, SWING, veya POSITION

# Eğitilecek semboller
SYMBOLS = ["BTC", "ETH"]  # Başlangıç için 2 sembol

# Konfigürasyonu göster
trade_config = TRADE_TYPE_CONFIGS[TRADE_TYPE]
print(f"Trade Type: {TRADE_TYPE.value}")
print(f"Data Interval: {trade_config.data_interval}")
print(f"Lookback Days: {trade_config.lookback_days}")
print(f"Encoder Length: {trade_config.min_encoder_length} - {trade_config.max_encoder_length}")
print(f"Prediction Length: {trade_config.min_prediction_length} - {trade_config.max_prediction_length}")
print(f"Min Training Samples: {trade_config.min_training_samples}")

In [ ]:
# Training config oluştur
config = TrainingConfig(
    symbols=SYMBOLS,
    trade_type=TRADE_TYPE,
    max_epochs=100,
    batch_size=64,
    use_gpu=True,
    num_workers=2,  # Colab için 2 worker yeterli
    n_optuna_trials=20,  # Hızlı başlangıç için düşük trial
)

print(f"Config: {config}")

## 📥 Veri Çekme

In [ ]:
from training.trainer import TFTTrainer

trainer = TFTTrainer(config)

# Step 1: Veri çek
print("Step 1: Fetching data from Binance...")
raw_data = await trainer.fetch_data()
print(f"Fetched {len(raw_data)} symbols")
for symbol, df in raw_data.items():
    print(f"  {symbol}: {len(df)} candles")

In [ ]:
# Step 2: Feature engineering
print("Step 2: Feature engineering...")
processed_data, metadata = trainer.engineer_features()
print(f"Total samples: {len(processed_data)}")
print(f"Features: {len(metadata['time_varying_unknown'])}")

## 🔧 Hyperparameter Optimization (Opsiyonel)

In [ ]:
# Step 3: Hyperparameter optimization (opsiyonel - uzun sürer)
SKIP_OPTUNA = True  # İlk eğitim için True, sonra False yapılabilir

if not SKIP_OPTUNA:
    print("Step 3: Hyperparameter optimization...")
    optuna_results = trainer.optimize_hyperparameters(n_trials=20)
    print(f"Best params: {optuna_results.get('model_config', {})}")
else:
    print("Step 3: Skipping Optuna (using default params)")

## 🏋️ Model Eğitimi

In [ ]:
# Step 5: Train final model
print("Step 5: Training final model...")
model = trainer.train_final_model()
print(f"Model trained! Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Step 7: Model kaydet
print("Step 7: Saving model...")
model_path = trainer.save_model()
print(f"Model saved to: {model_path}")

## 📤 Model Export

In [ ]:
# Google Drive'a kaydet
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(model_path, '/content/drive/MyDrive/')
print(f"Model copied to Google Drive!")

In [ ]:
# Alternatif: Dosyayı indir
from google.colab import files
files.download(model_path)

## 🔄 Full Pipeline (Otomatik)

In [ ]:
# Alternatif: Tam pipeline tek seferde
# results = await trainer.run_full_pipeline(
#     skip_optuna=True,
#     skip_validation=True,
#     skip_backtest=True
# )
# print(f"Results: {results}")